# 06 — 增加时间依赖：从 GMM 到贝叶斯滤波
回到第 02 章的三个高斯簇。观测空间、三个中心、观测噪声均不变，但现在隐变量不再独立采样，而是按照转移概率生成。必须重新生成有时间顺序的数据。

主动定义初始概率 $p(x_0)$、转移矩阵 $A_{ij}=p(x_t=j|x_{t-1}=i)$ 和观测密度 $p(y_t|x_t)$。本章先假设参数已知，只做推断。真实类别仅用于检查模拟结果，不进入滤波算法。

递推预测：$p(x_t|y_{0:t-1})=\sum_i p(x_t|x_{t-1}=i)p(x_{t-1}=i|y_{0:t-1})$。

更新：$p(x_t|y_{0:t})\propto p(y_t|x_t)p(x_t|y_{0:t-1})$。归一化常数就是当前观测的预测密度。

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.special import logsumexp
import matplotlib.pyplot as plt
rng = np.random.default_rng(42)
mu = np.array([[-2.,0.],[2.,0.],[0.,3.]])
# 增加噪声，以使时间信息的作用更明显。
sigma = 1.1
A = np.full((3,3),.025); np.fill_diagonal(A,.95)
T = 300
x = np.empty(T,dtype=int); x[0]=rng.integers(3)
for t in range(1,T): x[t]=rng.choice(3,p=A[x[t-1]])
y=mu[x]+sigma*rng.normal(size=(T,2))
log_cond=norm.logpdf(y[:,None,:],mu,sigma).sum(-1)
# 不使用时间信息：与第 02 章的 GMM 后验完全相同。
static=np.exp(log_cond-logsumexp(log_cond,axis=1,keepdims=True))
def filter_sequence(log_cond, A):
    posterior=np.ones(A.shape[0])/A.shape[0]
    filtered=[]; log_evidence=[]
    for t, lc in enumerate(log_cond):
        predicted=posterior if t==0 else posterior@A
        log_joint=np.log(predicted)+lc
        log_z=logsumexp(log_joint)
        posterior=np.exp(log_joint-log_z)
        filtered.append(posterior); log_evidence.append(log_z)
    return np.array(filtered),np.array(log_evidence)
filtered, ll = filter_sequence(log_cond,A)
assert np.allclose(filtered.sum(1),1)
print('static accuracy:', (static.argmax(1)==x).mean())
print('filter accuracy:', (filtered.argmax(1)==x).mean())
fig,ax=plt.subplots(figsize=(10,3))
ax.plot(x[:100],label='truth'); ax.plot(static.argmax(1)[:100],'.',label='static'); ax.plot(filtered.argmax(1)[:100],'--',label='filter'); ax.legend(); plt.show()

## 滤波与参数学习的连接
滤波本身不估计转移参数，但每个时刻得到的归一化常数 $p(y_t|y_{0:t-1})$ 相乘就是序列似然。因此可以直接比较参数，甚至优化它。下面只估计自转移概率 $a$，其他参数固定。

In [ ]:
candidates=np.linspace(.34,.995,100)
likelihood=[]
for a in candidates:
    candidate=np.full((3,3),(1-a)/2); np.fill_diagonal(candidate,a)
    likelihood.append(filter_sequence(log_cond,candidate)[1].sum())
print('MLE of self-transition probability (grid):', candidates[np.argmax(likelihood)])
plt.plot(candidates,likelihood); plt.axvline(.95,ls='--',label='simulation truth'); plt.legend(); plt.xlabel('a'); plt.ylabel('sequence log likelihood'); plt.show()

改变观测噪声、转移概率，观察静态后验和滤波后验的差异。离散隐变量使预测中的边缘化变成有限求和；连续线性高斯模型则可用卡尔曼滤波实现同一个递推原理。